In [1]:
%pip install "pydantic-ai-slim[google]" ddgs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pydantic_ai import Agent, BinaryContent
from pydantic_ai.models.google import GoogleModel
from pydantic import BaseModel, Field
from pydantic_ai.providers.google import GoogleProvider
from pydantic_ai.capabilities import WebSearch

GOOGLE_API_KEY = os.getenv("gemini")

class ImageAnalysisResult(BaseModel):
    summary: str = Field(description="이미지 전체 요약")
    objects: list[str] = Field(description="이미지에서 감지된 주요 객체 목록")
    category: str = Field(description="이미지의 분류 카테고리")


In [5]:
# 2. LangChain과 동일하게 503/429 발생 시 최대 6회 자동 재시도하도록 설정
provider = GoogleProvider(
    api_key=GOOGLE_API_KEY
)

model = GoogleModel(model_name='gemini-3.1-flash-lite', provider=provider)

# 에이전트 생성
agent = Agent(
    model,
    instructions="햇갈리면 웹검색을 쓸것 꼭 쓸 필요는 없음",
    capabilities=[WebSearch(native=False, local='duckduckgo')],
    output_type=ImageAnalysisResult
)

with open("sample_image.jpg", "rb") as f:
    image_bytes = f.read()
result = await agent.run(
[
    "이 이미지를 분석해라",
    BinaryContent(data=image_bytes, media_type="image/jpeg")
]
)

result.output

ImageAnalysisResult(summary='검은색 배경 위에 놓인 진한 핑크색(자홍색) 맥(MAC) 립스틱입니다. 립스틱 케이스는 은색과 검은색 조합의 전형적인 MAC 디자인입니다.', objects=['MAC lipstick'], category='beauty product')

In [6]:
output_data = result.output
print(output_data.summary)
print(output_data.objects)
print(output_data.category)

검은색 배경 위에 놓인 진한 핑크색(자홍색) 맥(MAC) 립스틱입니다. 립스틱 케이스는 은색과 검은색 조합의 전형적인 MAC 디자인입니다.
['MAC lipstick']
beauty product
